# Notebook 01 — Data Preparation
**Source:** `sales-dashboard/output/df_master.parquet`  
**Tujuan:** Agregasi dari level item → level order, siap untuk cohort & RFM.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Path relatif dari notebooks/ ke output proyek sebelumnya
SRC = Path('../../sales-dashboard/output/df_master.parquet')
OUT = Path('../output')
OUT.mkdir(exist_ok=True)

print('Source exists:', SRC.exists())

## 1. Load df_master

In [ ]:
df = pd.read_parquet(SRC)
df['order_month'] = df['order_month'].astype('period[M]')
print(f'df_master: {len(df):,} baris (level item)')
df.head(3)

## 2. Agregasi ke Level Order

In [ ]:
df_orders = (
    df.groupby('order_id')
    .agg(
        customer_id    = ('customer_id', 'first'),
        customer_state = ('customer_state', 'first'),
        order_date     = ('order_purchase_timestamp', 'min'),
        order_revenue  = ('revenue', 'sum'),
        n_items        = ('product_id', 'count')
    )
    .reset_index()
)

df_orders['order_month'] = df_orders['order_date'].dt.to_period('M')

print(f'df_orders: {len(df_orders):,} order unik')
print(f'Customer unik: {df_orders["customer_id"].nunique():,}')
assert df_orders['order_id'].nunique() == len(df_orders), 'Ada duplikat order_id!'
print('Validasi: tidak ada duplikat order_id.')

## 3. Statistik Dasar

In [ ]:
print('Periode data  :', df_orders['order_date'].min().date(),
      '→', df_orders['order_date'].max().date())
print('Total revenue : R$', f"{df_orders['order_revenue'].sum():,.0f}")
print('AOV (per order):', f"R$ {df_orders['order_revenue'].mean():,.2f}")
print('\nDistribusi orders per customer:')
orders_per_cust = df_orders.groupby('customer_id')['order_id'].count()
print(orders_per_cust.value_counts().head(5))

## 4. Simpan ke Parquet

In [ ]:
df_orders.to_parquet(OUT / 'df_orders.parquet', index=False)
print(f'Tersimpan: df_orders.parquet — {len(df_orders):,} baris, {df_orders.shape[1]} kolom')